# Demo Titanic -> Elasticsearch -> Kibana

Objectif:
- charger un CSV Titanic local
- indexer les documents dans Elasticsearch
- produire des stats exploitables dans Kibana


In [2]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "elasticsearch>=8,<9", "pandas"], check=True)
print("Dependencies installed")


Dependencies installed


In [3]:
import os
import pandas as pd
from elasticsearch import Elasticsearch, helpers

ES_URL = os.getenv("COURSE_ES_URL", "http://elasticsearch:9200")
DATA_DIR = os.getenv("COURSE_DATA_DIR", "/home/jovyan/data")
CSV_PATH = os.path.join(DATA_DIR, "titanic_sample.csv")
INDEX_NAME = "titanic_data"

print("ES_URL:", ES_URL)
print("CSV_PATH:", CSV_PATH)


ES_URL: http://elasticsearch:9200
CSV_PATH: /home/jovyan/data/titanic_sample.csv


In [11]:
df = pd.read_csv(CSV_PATH)

# Normalize booleans
bool_cols = ["adult_male", "alone"]
for col in bool_cols:
    df[col] = df[col].astype(str).str.strip().str.lower().map({"true": True, "false": False})

# Normalize numeric columns with nullable dtypes
int_cols = ["survived", "pclass", "sibsp", "parch"]
for col in int_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

float_cols = ["age", "fare"]
for col in float_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Build JSON-safe records (replace NaN/NA with None, numpy scalars -> Python scalars)
records = []
for rec in df.to_dict(orient="records"):
    clean = {}
    for k, v in rec.items():
        if pd.isna(v):
            clean[k] = None
        elif hasattr(v, "item"):
            clean[k] = v.item()
        else:
            clean[k] = v
    records.append(clean)

print(f"Loaded {len(records)} rows")
df.head()


Loaded 30 rows


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [12]:
es = Elasticsearch(ES_URL, request_timeout=30)
print("Elasticsearch version:", es.info()["version"]["number"])

mapping = {
    "properties": {
        "survived": {"type": "integer"},
        "pclass": {"type": "integer"},
        "sex": {"type": "keyword"},
        "age": {"type": "float"},
        "sibsp": {"type": "integer"},
        "parch": {"type": "integer"},
        "fare": {"type": "float"},
        "embarked": {"type": "keyword"},
        "class": {"type": "keyword"},
        "who": {"type": "keyword"},
        "adult_male": {"type": "boolean"},
        "deck": {"type": "keyword"},
        "embark_town": {"type": "keyword"},
        "alive": {"type": "keyword"},
        "alone": {"type": "boolean"}
    }
}

if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

es.indices.create(index=INDEX_NAME, mappings=mapping)

actions = [
    {"_index": INDEX_NAME, "_id": i + 1, "_source": rec}
    for i, rec in enumerate(records)
]

success, errors = helpers.bulk(es, actions, raise_on_error=False)
if errors:
    print("First bulk error:", errors[0])
    raise RuntimeError(f"{len(errors)} docs failed during bulk indexing")

es.indices.refresh(index=INDEX_NAME)
print(f"Indexed {success} docs into '{INDEX_NAME}'")


Elasticsearch version: 8.10.2
Indexed 30 docs into 'titanic_data'


In [13]:
query = {
    "size": 0,
    "aggs": {
        "survival_rate": {"avg": {"field": "survived"}},
        "by_sex": {"terms": {"field": "sex"}},
        "by_class": {"terms": {"field": "class"}},
        "avg_fare": {"avg": {"field": "fare"}}
    }
}

res = es.search(index=INDEX_NAME, body=query)
res["aggregations"]


{'survival_rate': {'value': 0.5333333333333333},
 'avg_fare': {'value': 30.667636553446453},
 'by_class': {'doc_count_error_upper_bound': 0,
  'sum_other_doc_count': 0,
  'buckets': [{'key': 'Third', 'doc_count': 15},
   {'key': 'First', 'doc_count': 8},
   {'key': 'Second', 'doc_count': 7}]},
 'by_sex': {'doc_count_error_upper_bound': 0,
  'sum_other_doc_count': 0,
  'buckets': [{'key': 'female', 'doc_count': 15},
   {'key': 'male', 'doc_count': 15}]}}

In [14]:
df.groupby("sex", dropna=False)["survived"].mean().rename("survival_rate")


sex
female    0.933333
male      0.133333
Name: survival_rate, dtype: Float64

## Kibana

1. Ouvrir Kibana: `http://localhost:5601`
2. Creer Data View: `titanic_data`
3. Dans Lens, essayer:
- Pie chart: repartition par `sex`
- Bar chart: moyenne `survived` par `class`
- Metric: moyenne `fare`


In [15]:
!pip install requests

In [17]:
import requests
import json

url = "http://elasticsearch:9200/shakespeare/_search"

response = requests.get(url)

data = response.json()
print(json.dumps(data, indent=2))

{
  "took": 0,
  "timed_out": false,
  "_shards": {
    "total": 1,
    "successful": 1,
    "skipped": 0,
    "failed": 0
  },
  "hits": {
    "total": {
      "value": 1,
      "relation": "eq"
    },
    "max_score": 1.0,
    "hits": [
      {
        "_index": "shakespeare",
        "_id": "1",
        "_score": 1.0,
        "_source": {
          "play_name": "Hamlet",
          "speaker": "HAMLET",
          "text_entry": "To be, or not to be: that is the question."
        }
      }
    ]
  }
}
